In [1]:
import subprocess
import sys

print("Installing dependencies...")
print("This will take about 5 minutes.\n")

# Core dependencies
packages = [
    "torch torchvision torchaudio",
    "transformers",
    "librosa",
    "ipywidgets",
    "joblib",
    "numpy",
    "scipy",
    "pillow",
    "opencv-python",
    "scikit-learn",
    "kornia"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkg.split(), check=False)

print("\nAll dependencies installed!")
print("Now run Cell 2 to setup models.")

Installing dependencies...
This will take about 5 minutes.

Installing torch torchvision torchaudio...
Installing transformers...
Installing librosa...
Installing ipywidgets...
Installing joblib...
Installing numpy...
Installing scipy...
Installing pillow...
Installing opencv-python...
Installing scikit-learn...
Installing kornia...

All dependencies installed!
Now run Cell 2 to setup models.


In [2]:
import os
import subprocess
import shutil
import torch
from pathlib import Path
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Wav2Vec2FeatureExtractor

print("="*60)
print(" EMOTION SYNTHESIS STUDIO - MODEL SETUP")
print("="*60)
print("This will take 10-15 minutes. Please be patient.\n")

# ============================================================================
# FUNCTION DEFINITIONS
# ============================================================================

def setup_wav2vec2(model_name="facebook/wav2vec2-base-960h"):
    """Setup Wav2Vec2 model for speech recognition"""
    print("🎤 STEP 1/3: Setting up Wav2Vec2 model...")
    try:
        wav2vec_model = Wav2Vec2ForCTC.from_pretrained(model_name)
        wav2vec_processor = Wav2Vec2Processor.from_pretrained(model_name)
        wav2vec_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        wav2vec_model = wav2vec_model.to(device)
        wav2vec_model.eval()
        print(f"Wav2Vec2 ready! Using device: {device}\n")
        return wav2vec_model, wav2vec_processor, wav2vec_extractor, device
    except Exception as e:
        print(f" Wav2Vec2 model loading issue: {e}")
        return None, None, None, None


def setup_sadtalker():
    """Setup SadTalker for video generation"""
    print("🎨 STEP 2/3: Setting up SadTalker...")
    os.chdir("/content")

    if os.path.exists('SadTalker'):
        subprocess.run(['rm', '-rf', 'SadTalker'], check=True)

    subprocess.run(['git', 'clone', 'https://github.com/OpenTalker/SadTalker.git'], check=True)
    os.chdir('SadTalker')

    print("   Installing SadTalker dependencies...")

    # Install required packages FIRST
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yacs'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imageio-ffmpeg'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'realesrgan'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'basicsr'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'facexlib'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gfpgan'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydub'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'safetensors'], check=False)

    # Apply patches IMMEDIATELY after installing basicsr
    print("   Applying compatibility patches...")

    # Patch 1: Fix basicsr (CRITICAL - do this before downloading models)
    possible_paths = [
        "/usr/local/lib/python3.12/dist-packages/basicsr",
        "/usr/local/lib/python3.11/dist-packages/basicsr",
        "/usr/local/lib/python3.10/dist-packages/basicsr"
    ]
    for p in possible_paths:
        check_file = os.path.join(p, 'data', 'degradations.py')
        if os.path.exists(check_file):
            with open(check_file, 'r') as f:
                code = f.read()
            if 'from torchvision.transforms.functional_tensor import rgb_to_grayscale' in code:
                code = code.replace(
                    'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                    'from torchvision.transforms.functional import rgb_to_grayscale'
                )
                with open(check_file, 'w') as f:
                    f.write(code)
                print("      ✓ Basicsr patched")
                break

    # NOW download models
    if not os.path.exists('./checkpoints'):
        subprocess.run(['bash', 'scripts/download_models.sh'], check=True)

    # Patch 2: Fix math logic
    target_file = "/content/SadTalker/src/face3d/util/preprocess.py"
    if os.path.exists(target_file):
        with open(target_file, 'r') as f:
            code = f.read()
        bad_line = "trans_params = np.array([w0, h0, s, t[0], t[1]])"
        new_line = "trans_params = np.array([float(w0), float(h0), float(s), float(t[0]), float(t[1])])"
        if bad_line in code:
            code = code.replace(bad_line, new_line)
            with open(target_file, 'w') as f:
                f.write(code)
            print("      ✓ Math logic patched")

    print("SadTalker ready!\n")


def setup_wav2lip():
    """Setup Wav2Lip for lip syncing"""
    print("STEP 3/3: Setting up Wav2Lip...")
    os.chdir("/content")

    # Force fresh clone
    if os.path.exists("Wav2Lip"):
        print("   Removing old Wav2Lip...")
        shutil.rmtree("Wav2Lip")

    print("   Cloning Wav2Lip...")
    os.system("git clone -q https://github.com/Rudrabha/Wav2Lip.git")

    os.chdir("/content/Wav2Lip")

    # Create directories
    os.system("mkdir -p checkpoints")
    os.system("mkdir -p face_detection/detection/sfd")

    # Download models using the reliable sources
    print("   Downloading Wav2Lip models...")

    checkpoint_path = Path("checkpoints/wav2lip_gan.pth")
    if not checkpoint_path.exists() or checkpoint_path.stat().st_size < 1000000:
        print("      Downloading wav2lip_gan.pth...")
        os.system('wget -q -O checkpoints/wav2lip_gan.pth "https://github.com/anothermartz/Easy-Wav2Lip/releases/download/Prerequesits/Wav2Lip_GAN.pth"')

    sfd_path = Path("face_detection/detection/sfd/s3fd.pth")
    if not sfd_path.exists() or sfd_path.stat().st_size < 1000000:
        print("      Downloading s3fd.pth...")
        os.system("wget -q https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth -O face_detection/detection/sfd/s3fd.pth")

    # Patch Wav2Lip for PyTorch 2.x compatibility
    print("   Patching Wav2Lip for PyTorch 2.x...")

    # Patch inference.py with precise line replacements
    inference_file = Path("/content/Wav2Lip/inference.py")
    with open(inference_file, 'r') as f:
        code = f.read()

    # Replace the specific _load function
    old_load_func = """def _load(checkpoint_path):
  if device == 'cuda':
    checkpoint = torch.load(checkpoint_path)
  else:
    checkpoint = torch.load(checkpoint_path,
                map_location=lambda storage, loc: storage)
  return checkpoint"""

    new_load_func = """def _load(checkpoint_path):
  if device == 'cuda':
    checkpoint = torch.load(checkpoint_path, weights_only=False)
  else:
    checkpoint = torch.load(checkpoint_path,
                map_location=lambda storage, loc: storage, weights_only=False)
  return checkpoint"""

    code = code.replace(old_load_func, new_load_func)

    with open(inference_file, 'w') as f:
        f.write(code)
    print("      ✓ inference.py patched")

    # Patch models/wav2lip.py
    models_file = Path("/content/Wav2Lip/models/wav2lip.py")
    if models_file.exists():
        with open(models_file, 'r') as f:
            code = f.read()

        if 'torch.load(' in code and 'weights_only' not in code:
            import re
            code = re.sub(
                r'torch\.load\(([^,\)]+)\)',
                r'torch.load(\1, weights_only=False)',
                code
            )
            with open(models_file, 'w') as f:
                f.write(code)
            print("      ✓ models/wav2lip.py patched")

    # Patch face detection
    face_detect_file = Path("/content/Wav2Lip/face_detection/detection/sfd/detect.py")
    if face_detect_file.exists():
        with open(face_detect_file, 'r') as f:
            code = f.read()

        if 'torch.load(' in code and 'weights_only' not in code:
            import re
            code = re.sub(
                r'torch\.load\(([^,\)]+)\)',
                r'torch.load(\1, weights_only=False)',
                code
            )
            code = re.sub(
                r'torch\.load\(([^,\)]+),\s*map_location=([^,\)]+)\)',
                r'torch.load(\1, map_location=\2, weights_only=False)',
                code
            )
            with open(face_detect_file, 'w') as f:
                f.write(code)
            print("      ✓ face detection patched")

    print("Wav2Lip ready!\n")


# ============================================================================
# RUN SETUP
# ============================================================================

# Setup all models
wav2vec_model, wav2vec_processor, wav2vec_extractor, device = setup_wav2vec2()
setup_sadtalker()
setup_wav2lip()

print("="*60)
print("ALL MODELS READY!")
print("="*60)
print("Now run Cell 3 to launch the interactive UI.\n")

🚀 EMOTION SYNTHESIS STUDIO - MODEL SETUP
This will take 10-15 minutes. Please be patient.

🎤 STEP 1/3: Setting up Wav2Vec2 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

✅ Wav2Vec2 ready! Using device: cuda

🎨 STEP 2/3: Setting up SadTalker...
   Installing SadTalker dependencies...
   Applying compatibility patches...
      ✓ Basicsr patched
      ✓ Math logic patched
✅ SadTalker ready!

STEP 3/3: Setting up Wav2Lip...
   Cloning Wav2Lip...
   Patching Wav2Lip for PyTorch 2.x...
      ✓ inference.py patched
Wav2Lip ready!

ALL MODELS READY!
Now run Cell 3 to launch the interactive UI.



In [3]:
# ============================================================================
# INTERACTIVE EMOTION SYNTHESIS UI - FINAL WORKING CODE
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Audio as IPAudio, Image as IPImage
import subprocess
import sys
import os
from pathlib import Path
import time
import numpy as np
import librosa
from base64 import b64encode
import joblib

# ============================================================================
# GLOBAL VARIABLES AND MODEL PATHS
# ============================================================================
BASE_DIR = Path("/content/emotion_project")
BASE_DIR.mkdir(exist_ok=True)
(BASE_DIR / "outputs").mkdir(exist_ok=True)

uploaded_image = None
uploaded_audio = None
detected_emotion = None
detected_intensity = None
emotion_model = None
intensity_model = None

# Custom Model File Paths (Confirmed to be in /content/)
EMOTION_MODEL_PATH = "/content/emotion_model.pkl"
INTENSITY_MODEL_PATH = "/content/intensity_model.pkl"

# ============================================================================
# FEATURE EXTRACTION
# ============================================================================
def extract_features(speech, sr=16000, n_mfcc=40):
    """Extracts 40 MFCCs (Standard assumption for SER models)."""
    try:
        mfccs = librosa.feature.mfcc(y=speech, sr=sr, n_mfcc=n_mfcc)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        return mfccs_mean.reshape(1, -1)
    except Exception as e:
        print(f"Feature extraction failed: {e}")
        return None

# ============================================================================
# EMOTION DETECTION FUNCTION
# ============================================================================
def load_emotion_detector():
    """Load custom emotion and intensity classifiers using joblib"""
    global emotion_model, intensity_model

    if emotion_model is not None and intensity_model is not None:
        return True

    if not os.path.exists(EMOTION_MODEL_PATH) or not os.path.exists(INTENSITY_MODEL_PATH):
        print("ERROR: Custom model files (.pkl) not found. Check file names.")
        return False

    try:
        print("Loading custom emotion and intensity models...")
        emotion_model = joblib.load(EMOTION_MODEL_PATH)
        intensity_model = joblib.load(INTENSITY_MODEL_PATH)
        print(" Custom models ready.")
        return True
    except Exception as e:
        print(f" Failed to load custom models: {e}")
        return False

def detect_emotion_from_audio(audio_path):
    """Detect emotion and intensity using custom joblib models"""
    global emotion_model, intensity_model

    if emotion_model is None or intensity_model is None:
        if not load_emotion_detector():
            return None, None, None

    try:
        speech, sr = librosa.load(audio_path, sr=16000)
        features = extract_features(speech, sr)
        if features is None:
             return None, None, None

        # Prediction
        predicted_emotion = emotion_model.predict(features)[0]
        predicted_intensity = intensity_model.predict(features)[0]

        # Confidence calculation
        confidence = np.max(emotion_model.predict_proba(features)) if hasattr(emotion_model, 'predict_proba') else 0.0

        return predicted_emotion, predicted_intensity, confidence

    except Exception as e:
        print(f" Custom model detection failed: {e}")
        return None, None, None

# ============================================================================
# VIDEO GENERATION FUNCTION
# ============================================================================
def generate_emotion_video(image_path, audio_path, emotion, intensity, output_name="result"):
    """Generate emotion-controlled talking face video with intensity control"""

    print(f"\n{'='*60}")
    print(f"🎬 GENERATING VIDEO: {emotion.upper()} ({intensity.upper()})")
    print(f"{'='*60}\n")

    emotion_params = {
        'happy': {'pose_style': 1, 'expression_scale': 1.56, 'still': False},
        'sad': {'pose_style': 0, 'expression_scale': 0.96, 'still': True},
        'angry': {'pose_style': 2, 'expression_scale': 1.92, 'still': False},
        'fearful': {'pose_style': 1, 'expression_scale': 1.44, 'still': False},
        'surprised': {'pose_style': 2, 'expression_scale': 1.80, 'still': False},
        'neutral': {'pose_style': 0, 'expression_scale': 0.60, 'still': True},
        'calm': {'pose_style': 0, 'expression_scale': 0.72, 'still': True},
        'disgust': {'pose_style': 1, 'expression_scale': 1.56, 'still': False}
    }

    params = emotion_params.get(emotion, emotion_params['neutral']).copy()

    if intensity == 'high':
        params['expression_scale'] *= 1.25
    elif intensity == 'low':
        params['expression_scale'] *= 0.75

    try:
        # STAGE 1: SadTalker
        print("    Stage 1: SadTalker (Generating animated face)...")
        start_time = time.time()
        os.chdir("/content/SadTalker")
        sadtalker_output = BASE_DIR / "outputs" / f"temp_sadtalker_{output_name}"
        sadtalker_output.mkdir(parents=True, exist_ok=True)

        launcher_code = f"""
import os, sys, warnings, runpy
import numpy as np
np.float = np.float64; np.int = np.int_; np.bool = np.bool_; np.object = object
np.str = str; np.complex = complex
class VisibleDeprecationWarning(UserWarning): pass
np.VisibleDeprecationWarning = VisibleDeprecationWarning

sys.argv = [
    'inference.py', '--driven_audio', '{audio_path}', '--source_image', '{image_path}',
    '--result_dir', '{sadtalker_output}', '--pose_style', '{params["pose_style"]}',
    '--expression_scale', '{params["expression_scale"]}',
    {f"'--still'," if params['still'] else ""}'--enhancer', 'gfpgan',
    '--preprocess', 'full', '--size', '256'
]
runpy.run_path('inference.py', run_name='__main__')
"""
        with open("launcher_ui.py", "w") as f: f.write(launcher_code)
        process = subprocess.run([sys.executable, "launcher_ui.py"], capture_output=True, text=True)

        if process.returncode != 0: raise Exception(f"SadTalker failed: {process.stderr[-300:]}")
        sadtalker_videos = list(sadtalker_output.glob("**/*.mp4"))
        if not sadtalker_videos: raise Exception("SadTalker produced no output")
        sadtalker_video = str(sadtalker_videos[0])
        sadtalker_time = time.time() - start_time
        print(f"       Done ({sadtalker_time:.1f}s)")

        # STAGE 2: Wav2Lip
        print("  Stage 2: Wav2Lip (Syncing lips)...")
        start_time_w2l = time.time()
        os.chdir("/content/Wav2Lip")
        temp_audio = BASE_DIR / f"temp_audio_{output_name}.wav"
        subprocess.run(['ffmpeg', '-y', '-i', audio_path, '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', str(temp_audio)], capture_output=True, check=True)
        final_video = BASE_DIR / "outputs" / f"{output_name}_{emotion}_{intensity}_FINAL.mp4"

        cmd = [
            sys.executable, 'inference.py',
            '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
            '--face', sadtalker_video,
            '--audio', str(temp_audio),
            '--outfile', str(final_video),
            '--pads', '0', '10', '0', '0',
            '--fps', '25',
            '--resize_factor', '1'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0 or not final_video.exists(): raise Exception(f"Wav2Lip error: {result.stderr[-300:]}")

        wav2lip_time = time.time() - start_time_w2l
        file_size = final_video.stat().st_size / 1024 / 1024

        print(f"      Done ({wav2lip_time:.1f}s, {file_size:.2f}MB)")

        # Cleanup temp files
        if temp_audio.exists(): temp_audio.unlink()

        print(f"\nSUCCESS! Video saved: {final_video.name}")
        print(f"   Total time: {sadtalker_time + wav2lip_time:.1f}s\n")

        return str(final_video)

    except Exception as e:
        print(f"\nFAILED: {str(e)}\n")
        return None

print("✅ Core functions loaded successfully!")

✅ Core functions loaded successfully!


In [ ]:
# Patch audio.py for librosa compatibility
from pathlib import Path

audio_file = Path("/content/Wav2Lip/audio.py")
if audio_file.exists():
    with open(audio_file, 'r') as f:
        code = f.read()

    # Fix librosa.filters.mel() call
    code = code.replace(
        'librosa.filters.mel(hp.sample_rate, hp.n_fft,',
        'librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,'
    )
    with open(audio_file, 'w') as f:
        f.write(code)
    print("✓ audio.py patched for librosa compatibility!")

In [6]:
# ============================================================================
# UI COMPONENTS AND HANDLERS - Minimal & Academic
# ============================================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from base64 import b64encode
from pathlib import Path
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics as skm
import io
import os

# Globals (assumes BASE_DIR and functions like detect_emotion_from_audio and generate_emotion_video exist)
BASE_DIR = Path("/content/emotion_project")
BASE_DIR.mkdir(exist_ok=True)
(BASE_DIR / "outputs").mkdir(exist_ok=True)

uploaded_image = None
uploaded_audio = None
detected_emotion = None
detected_intensity = None

# UI widgets
output_area = widgets.Output()

title_html = """
<div style="text-align: center; padding: 14px; border-radius: 8px; margin-bottom: 12px;">
    <h1 style="margin: 0; font-size: 28px;">Emotion Synthesis Studio</h1>
    <p style="margin-top: 6px; color: #333; font-size: 14px;">Upload face image and audio, then generate an emotion-controlled avatar</p>
</div>
"""

image_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Upload face image', layout=widgets.Layout(width='300px'))
audio_upload = widgets.FileUpload(accept='audio/*', multiple=False, description='Upload audio', layout=widgets.Layout(width='300px'))

# Mode values: 'auto' or 'manual' (no emoji)
emotion_mode = widgets.RadioButtons(options=[('Auto-detect emotion from audio', 'auto'), ('Manual selection', 'manual')],
                                   description='Mode:', disabled=False, style={'description_width': 'initial'})

emotion_dropdown = widgets.Dropdown(options=['happy', 'sad', 'angry', 'fearful', 'surprised', 'neutral', 'calm', 'disgust'],
                                    value='happy', description='Emotion:', disabled=True, layout=widgets.Layout(width='250px'))

intensity_dropdown = widgets.Dropdown(options=['normal', 'low', 'high'],
                                      value='normal', description='Intensity:', disabled=True, layout=widgets.Layout(width='250px'))

generate_btn = widgets.Button(description='Generate video', button_style='primary', layout=widgets.Layout(width='200px', height='40px'))
status_label = widgets.HTML(value='<p style="color: gray;">Upload image and audio to begin...</p>')
image_preview = widgets.HTML()
audio_preview = widgets.HTML()
emotion_preview = widgets.HTML()
video_preview = widgets.HTML()

# Optional evaluation CSV upload for report generation
eval_upload = widgets.FileUpload(accept='.csv', multiple=False, description='Upload evaluation CSV (optional)', layout=widgets.Layout(width='420px'))
report_btn = widgets.Button(description='Generate report from CSV', button_style='', layout=widgets.Layout(width='220px', height='36px'))

# Helper: write uploaded file to disk and return path
def _save_upload_to_file(upload_widget, out_path: Path):
    if len(upload_widget.value) == 0:
        return None
    uploaded = list(upload_widget.value.values())[0]
    with open(out_path, 'wb') as f:
        f.write(uploaded['content'])
    return str(out_path)

# Event handlers
def on_image_upload(change):
    global uploaded_image
    if len(image_upload.value) > 0:
        uploaded = list(image_upload.value.values())[0]
        image_path = BASE_DIR / "temp_uploaded_image.jpg"
        with open(image_path, 'wb') as f:
            f.write(uploaded['content'])
        uploaded_image = str(image_path)
        img_b64 = b64encode(uploaded['content']).decode()
        image_preview.value = f'''<div style="text-align: center; margin: 10px;"><p style="color: green; margin:0;">Image uploaded</p>
                                   <img src="data:image/jpeg;base64,{img_b64}" style="max-width:200px; border-radius:8px; border:1px solid #ccc;"></div>'''
        status_label.value = '<p style="color: #d2691e;">Now upload audio...</p>'
        check_ready()

def on_audio_upload(change):
    global uploaded_audio
    if len(audio_upload.value) > 0:
        uploaded = list(audio_upload.value.values())[0]
        audio_path = BASE_DIR / "temp_uploaded_audio.wav"
        with open(audio_path, 'wb') as f:
            f.write(uploaded['content'])
        uploaded_audio = str(audio_path)
        audio_preview.value = f'''<div style="text-align:center; margin:10px;"><p style="color: green; margin:0;">Audio uploaded</p>
                                   <audio controls style="width:300px;"><source src="data:audio/wav;base64,{b64encode(uploaded["content"]).decode()}" type="audio/wav"></audio></div>'''
        status_label.value = '<p style="color: #2f4f4f;">Choose emotion mode...</p>'
        check_ready()

def on_emotion_mode_change(change):
    mode = emotion_mode.value
    if mode == 'manual':
        emotion_dropdown.disabled = False
        intensity_dropdown.disabled = False
        emotion_preview.value = '<p style="color:#333; margin:0;">Select emotion and intensity manually</p>'
    else:
        emotion_dropdown.disabled = True
        intensity_dropdown.disabled = True
        emotion_preview.value = '<p style="color:#333; margin:0;">Emotion and intensity will be auto-detected from audio</p>'
    check_ready()

def check_ready():
    if uploaded_image and uploaded_audio:
        generate_btn.disabled = False
        status_label.value = '<p style="color: #006400; font-weight:600;">Ready to generate</p>'
    else:
        generate_btn.disabled = True

def on_generate_click(b):
    with output_area:
        clear_output()
        if not uploaded_image or not uploaded_audio:
            print("Please upload both image and audio first.")
            return

        final_emotion = None
        final_intensity = 'normal'

        if emotion_mode.value == 'auto':
            print("Detecting emotion and intensity from audio...")
            emotion, intensity, confidence = detect_emotion_from_audio(uploaded_audio)
            if emotion and intensity:
                final_emotion = emotion
                final_intensity = intensity
                emotion_preview.value = f'<div style="background:#f0f8ff; padding:10px; border-radius:8px; margin:8px;"><p style="margin:0; font-weight:600;">Detected: {emotion.upper()} ({intensity.upper()}) (confidence: {confidence:.2%})</p></div>'
                print(f"Detected emotion: {emotion} ({intensity}) (confidence: {confidence:.2%})")
            else:
                print("Detection failed. Using neutral/normal fallback.")
                final_emotion = 'neutral'
        else:
            final_emotion = emotion_dropdown.value
            final_intensity = intensity_dropdown.value
            print(f"Using manual selection: {final_emotion.upper()} ({final_intensity.upper()})")

        status_label.value = f'<p style="color:#8b4513;">Generating {final_emotion} ({final_intensity}) video... this may take a couple of minutes</p>'

        timestamp = int(time.time())
        video_path = generate_emotion_video(uploaded_image, uploaded_audio, final_emotion, final_intensity,
                                            output_name=f"user_{timestamp}")

        if video_path and Path(video_path).exists():
            with open(video_path, 'rb') as f:
                video_data = f.read()
            video_b64 = b64encode(video_data).decode()
            video_preview.value = f'''<div style="text-align:center; margin: 18px; padding: 18px; border-radius:8px; background:#f7f7f7;">
                                      <h3 style="margin:0 0 8px 0;">Generated video</h3>
                                      <video width="400" controls style="border-radius:6px; border:1px solid #ddd;">
                                        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
                                      </video>
                                      <p style="margin-top:8px; font-size:13px;">Saved as: {Path(video_path).name}</p>
                                      </div>'''
            status_label.value = '<p style="color:#006400; font-weight:600;">Video generated successfully</p>'
            print(f'To download programmatically: from google.colab import files; files.download("{video_path}")')
        else:
            status_label.value = '<p style="color:#8b0000;">Video generation failed. Check logs above.</p>'
            print("Video generation failed; check stdout/stderr for details.")

# Attach event handlers
image_upload.observe(on_image_upload, names='value')
audio_upload.observe(on_audio_upload, names='value')
emotion_mode.observe(on_emotion_mode_change, names='value')
generate_btn.on_click(on_generate_click)

# ---------- Reporting helpers ----------
def _plot_confusion_matrix(y_true, y_pred, labels, ax=None):
    cm = skm.confusion_matrix(y_true, y_pred, labels=labels)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,5))
    else:
        fig = None
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_title('Confusion matrix')
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=9)
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')
    if fig:
        plt.tight_layout()
        return fig
    return None

def _plot_class_distribution(y, ax=None):
    series = pd.Series(y).value_counts().sort_index()
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,3))
    else:
        fig = None
    series.plot(kind='bar', ax=ax)
    ax.set_title('Class distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    if fig:
        plt.tight_layout()
        return fig
    return None

def generate_report_from_csv(file_content_bytes):
    """
    Expected CSV minimal columns:
      - true : ground-truth labels (strings)
      - pred : model predictions (strings)
    Optional columns:
      - prob_<class> : predicted probabilities per class
      - model_a_pred, model_b_pred : for model comparison
    """
    try:
        df = pd.read_csv(io.BytesIO(file_content_bytes))
    except Exception as e:
        with output_area:
            print("Failed to read CSV:", e)
        return

    with output_area:
        clear_output()
        print("Report summary")
        print("--------------")
        # Basic check for columns
        if 'true' not in df.columns or 'pred' not in df.columns:
            print("CSV must contain 'true' and 'pred' columns for direct evaluation.")
            display(df.head())
            return

        y_true = df['true'].astype(str).values
        y_pred = df['pred'].astype(str).values
        labels = sorted(list(set(y_true) | set(y_pred)))

        # Overall accuracy
        acc = skm.accuracy_score(y_true, y_pred)
        print(f"Overall accuracy: {acc:.4f}")

        # Classification report
        print("\nClassification report:")
        print(skm.classification_report(y_true, y_pred, labels=labels, zero_division=0))

        # Confusion matrix plot
        fig_cm, ax_cm = plt.subplots(figsize=(6,5))
        _plot_confusion_matrix(y_true, y_pred, labels, ax=ax_cm)
        plt.show()

        # Class distribution
        fig_cd, ax_cd = plt.subplots(figsize=(6,3))
        _plot_class_distribution(y_true, ax=ax_cd)
        plt.show()

        # If there are probability columns, attempt to compute average confidence
        prob_cols = [c for c in df.columns if c.startswith('prob_')]
        if prob_cols:
            avg_conf = df[prob_cols].max(axis=1).mean()
            print(f"Average predicted confidence (from prob_* columns): {avg_conf:.3f}")

        # Model comparison table if present
        model_pred_cols = [c for c in df.columns if c.endswith('_pred') and c != 'pred']
        if model_pred_cols:
            compare_rows = []
            for col in ['pred'] + model_pred_cols:
                acc_i = skm.accuracy_score(y_true, df[col].astype(str).values)
                compare_rows.append({'model': col, 'accuracy': acc_i})
            compare_df = pd.DataFrame(compare_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
            print("\nModel comparison table")
            display(compare_df)

        # Simple conclusion text block
        print("\nConclusion and next steps")
        print("-------------------------")
        print("The table and plots above summarize the current evaluation. If accuracy is unsatisfactory consider:")
        print("- Re-training with more balanced data")
        print("- Feature engineering (additional audio features)")
        print("- Fine-tuning classifier hyperparameters")
        print("- Cross-validation to estimate variability")

# Report button handler
def on_report_click(b):
    if len(eval_upload.value) == 0:
        with output_area:
            clear_output()
            print("No evaluation CSV uploaded. Upload a CSV with 'true' and 'pred' columns and try again.")
        return
    uploaded = list(eval_upload.value.values())[0]
    generate_report_from_csv(uploaded['content'])

report_btn.on_click(on_report_click)

# ---------- Layout and display ----------
# Build display layout (minimal, academic style)
display(HTML(title_html))

upload_box = widgets.VBox([
    widgets.HTML('<h4 style="margin:0 0 6px 0;">Step 1: Upload files</h4>'),
    widgets.HBox([image_upload, audio_upload]),
    widgets.HBox([image_preview, audio_preview], layout=widgets.Layout(margin='6px 0 0 0'))
])

emotion_box = widgets.VBox([
    widgets.HTML('<h4 style="margin:10px 0 6px 0;">Step 2: Choose emotion</h4>'),
    emotion_mode,
    widgets.HBox([emotion_dropdown, intensity_dropdown], layout=widgets.Layout(margin='6px 0 0 0')),
    emotion_preview
])

generate_box = widgets.VBox([
    widgets.HTML('<h4 style="margin:10px 0 6px 0;">Step 3: Generate</h4>'),
    status_label,
    generate_btn
])

report_box = widgets.VBox([
    widgets.HTML('<h4 style="margin:10px 0 6px 0;">Optional: Evaluation and Report</h4>'),
    widgets.HBox([eval_upload, report_btn]),
    widgets.HTML('<div style="font-size:12px; color:#555; margin-top:6px;">Upload a CSV (columns: true,pred) to produce accuracy, confusion matrix and a short report.</div>')
])

main_ui = widgets.VBox([
    upload_box,
    widgets.HTML('<hr style="border:1px solid #eaeaea; margin:10px 0;">'),
    emotion_box,
    widgets.HTML('<hr style="border:1px solid #eaeaea; margin:10px 0;">'),
    generate_box,
    widgets.HTML('<hr style="border:1px solid #eaeaea; margin:10px 0;">'),
    report_box,
    widgets.HTML('<hr style="border:1px solid #eaeaea; margin:10px 0;">'),
    widgets.HTML('<div style="margin:6px 0 12px 0;"><strong>Generated video</strong></div>'),
    video_preview,
    output_area
], layout=widgets.Layout(padding='12px', border='1px solid #ddd', border_radius='8px'))

display(main_ui)

print("\n" + "="*60)
print("UI Ready")
print("="*60)
print("Upload image and audio, then choose auto or manual mode to begin.")
print("Optional: upload CSV for evaluation (columns: true,pred).")
print("="*60)



UI Ready
Upload image and audio, then choose auto or manual mode to begin.
Optional: upload CSV for evaluation (columns: true,pred).
